# Description-Based Rich-Text TF-IDF Recommender

----------------
**MSc Data Science Dissertation**  
**Author:** Vikrant Deshmukh  
**University:** University of Bristol  
**Project:** Video Game Recommendation System



This notebook explores an experimental lexical content-based recommender using cleaned Steam game descriptions. It was developed as an alternative text-based recommendation approach during model development and was not included in the final hybrid recommender.

The experimental model uses:

- cleaned descriptive text;
- TF-IDF unigram and bigram features;
- cosine nearest-neighbour retrieval; and
- ranked candidate generation for qualitative analysis.

## 1. Notebook Objectives

The notebook aims to:

1. construct a non-duplicated description field;
2. measure descriptive-text coverage across the catalogue;
3. build a TF-IDF representation of the eligible games;
4. retrieve lexically similar games using cosine distance; and
5. export ranked candidate lists for the same four case-study queries used by the other recommenders.


In [4]:
# Core libraries

import json
import os
import re
import unicodedata

import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

## 2. Dataset Loading

The notebook reuses the cleaned Steam Games Dataset 2025 produced during the data-preparation stage.

In [2]:
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
# Configure the project and dataset paths.

if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/MSC_DISSERTATION")
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "Data"
OUTPUT_DIR = PROJECT_ROOT / "Output/richtext_tf_idf_outputs"

data_path = DATA_DIR / "games_clean_v1.csv"

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at: {data_path}")

games = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Dataset shape:", games.shape)

Dataset loaded successfully.
Dataset shape: (89618, 50)


## 3. Required Column Validation

Only columns required by the description-based recommender are checked here. General dataset validation was completed during data preparation and metadata-model development.

In [ ]:
required_columns = {
    "appid",
    "display_name",
    "short_description_clean",
    "about_the_game_clean",
}

missing_required_columns = sorted(
    required_columns.difference(games.columns)
)

if missing_required_columns:
    raise KeyError(
        "Missing required columns: "
        + ", ".join(missing_required_columns)
    )

print("Required description columns are available.")

Required description columns are available.


## 4. Robust Game-Title Normalisation

Game titles may contain curly apostrophes, trademark symbols, punctuation or inconsistent capitalisation. A shared normalisation rule is used for title search and lookup.

In [8]:
def normalise_game_title(title):
    """Return a stable lookup key for a game title."""

    title = str(title)


    title = title.casefold()

    title = (
        title
        .replace("’", "'")
        .replace("‘", "'")
        .replace("`", "'")
        .replace("®", "")
        .replace("™", "")
    )

    title = unicodedata.normalize(
        "NFKD",
        title,
    )

    title = title.casefold()

    title = re.sub(
        r"[^a-z0-9]+",
        " ",
        title,
    )

    return re.sub(
        r"\s+",
        " ",
        title,
    ).strip()


games["normalised_display_name_rich_text"] = (
    games["display_name"]
    .apply(normalise_game_title)
)

print(
    games[
        [
            "display_name",
            "normalised_display_name_rich_text",
        ]
    ]
    .sample(10)
)

                          display_name normalised_display_name_rich_text
37328               Traps N' Gemstones                 traps n gemstones
41650                       Holy Shift                        holy shift
66168           Space Pirates for Life            space pirates for life
22264                     Final Knight                      final knight
32155                  里帰り | Satogaeri                         satogaeri
1386        MechWarrior 5: Mercenaries         mechwarrior 5 mercenaries
19602                      Cave Heroes                       cave heroes
29117  EreaDrone : FPV Drone Simulator     ereadrone fpv drone simulator
3743                     Trick & Treat                       trick treat
522                  Street Fighter™ 6                  street fighter 6


## 5. Description Construction

The final representation therefore uses:

- `about_the_game_clean` when it is available; and
- `short_description_clean` only as a fallback.

This avoids direct text duplication while retaining catalogue coverage.

In [9]:
about_text = (
    games["about_the_game_clean"]
    .fillna("")
    .astype(str)
    .str.strip()
)

short_text = (
    games["short_description_clean"]
    .fillna("")
    .astype(str)
    .str.strip()
)

games["description_text"] = np.where(
    about_text.ne(""),
    about_text,
    short_text,
)

games["description_text"] = (
    pd.Series(
        games["description_text"],
        index=games.index,
        dtype="object",
    )
    .fillna("")
    .astype(str)
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
    .str.strip()
)

games["description_word_count"] = (
    games["description_text"]
    .str.split()
    .str.len()
    .fillna(0)
    .astype(int)
)

print("Sample description:\n")
print(games.loc[0, "description_text"][:1000])

Sample description:

for over two decades counter strike has offered an elite competitive experience one shaped by millions of players from across the globe and now the next chapter in the cs story is about to begin this is counter strike 2 a free upgrade to cs go counter strike 2 marks the largest technical leap in counter strike s history built on the source 2 engine counter strike 2 is modernized with realistic physically based rendering state of the art networking and upgraded community workshop tools in addition to the classic objective focused gameplay that counter strike pioneered in 1999 counter strike 2 features all new cs ratings with the updated premier mode global and regional leaderboards upgraded and overhauled maps game changing dynamic smoke grenades tick rate independent gameplay redesigned visual effects and audio all items from cs go moving forward to cs2


## 6. Description Coverage Analysis

Very short descriptions provide little lexical evidence and may create unstable similarity estimates. The final model therefore requires at least 30 words.

The threshold preserves broad catalogue coverage while removing empty and extremely sparse descriptions.

In [11]:
MIN_DESCRIPTION_WORDS = 30

total_games = len(games)

non_empty_descriptions = (
    games["description_text"]
    .ne("")
    .sum()
)

eligible_mask = (
    games["description_word_count"]
    >= MIN_DESCRIPTION_WORDS
)

eligible_games = int(
    eligible_mask.sum()
)

excluded_games = int(
    total_games - eligible_games
)

coverage_percentage = (
    100 * eligible_games / total_games
)

print("Total games:", f"{total_games:,}")
print(
    "Non-empty descriptions:",
    f"{non_empty_descriptions:,}",
)
print(
    "Eligible descriptions:",
    f"{eligible_games:,}",
)
print(
    "Excluded descriptions:",
    f"{excluded_games:,}",
)
print(
    "Eligible catalogue coverage:",
    f"{coverage_percentage:.2f}%",
)

print("\nDescription word-count summary:")
print(
    games.loc[
        games["description_text"].ne(""),
        "description_word_count",
    ]
    .describe()
)

Total games: 89,618
Non-empty descriptions: 89,363
Eligible descriptions: 87,568
Excluded descriptions: 2,050
Eligible catalogue coverage: 97.71%

Description word-count summary:
count    89363.000000
mean       215.191836
std        170.073289
min          1.000000
25%        112.000000
50%        177.000000
75%        272.000000
max      11066.000000
Name: description_word_count, dtype: float64


In [12]:
description_games = (
    games.loc[eligible_mask]
    .copy()
    .reset_index(drop=True)
)

if description_games.empty:
    raise ValueError(
        "No games satisfy the description eligibility rule."
    )

print(
    "Description-model dataset shape:",
    description_games.shape,
)

Description-model dataset shape: (87568, 53)


## 7. TF-IDF Description Representation

The description text is converted into sparse TF-IDF vectors.

The configuration uses:

- unigrams and bigrams;
- English stop-word removal;
- Unicode accent normalisation;
- a 30,000-feature vocabulary cap;
- rare-term filtering with `min_df=3`;
- sublinear term frequency to reduce repeated marketing-language influence; and
- L2 normalisation for cosine-based retrieval.

In [13]:
RICH_TEXT_TFIDF_CONFIG = {
    "stop_words": "english",
    "strip_accents": "unicode",
    "max_features": 30000,
    "ngram_range": (1, 2),
    "min_df": 3,
    "max_df": 0.95,
    "sublinear_tf": True,
    "norm": "l2",
    "dtype": np.float32,
}

rich_text_tfidf = TfidfVectorizer(
    **RICH_TEXT_TFIDF_CONFIG
)

rich_text_matrix = rich_text_tfidf.fit_transform(
    description_games["description_text"]
)

print(
    "Rich-text TF-IDF matrix shape:",
    rich_text_matrix.shape,
)

print(
    "Vocabulary size:",
    f"{len(rich_text_tfidf.vocabulary_):,}",
)

Rich-text TF-IDF matrix shape: (87568, 30000)
Vocabulary size: 30,000


## 8. Cosine Nearest-Neighbour Model

A brute-force cosine nearest-neighbour model is fitted to the sparse TF-IDF matrix. Cosine distance is later converted into similarity using `1 - distance`.

In [14]:
rich_text_nn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)

rich_text_nn.fit(
    rich_text_matrix
)

print(
    "Rich-text nearest-neighbour model fitted successfully."
)

Rich-text nearest-neighbour model fitted successfully.


## 9. Eligible Title Mapping

Only games represented in the TF-IDF matrix can be queried. Duplicate normalised titles are resolved by retaining the record with the largest Steam recommendation count when that field is available.

In [15]:
description_games[
    "normalised_display_name_rich_text"
] = (
    description_games["display_name"]
    .apply(normalise_game_title)
)

if "recommendations" in description_games.columns:
    title_mapping_source = (
        description_games
        .sort_values(
            "recommendations",
            ascending=False,
            na_position="last",
        )
    )
else:
    title_mapping_source = description_games

rich_text_title_to_model_row = {}

for model_row, title_key in zip(
    title_mapping_source.index,
    title_mapping_source[
        "normalised_display_name_rich_text"
    ],
):
    if not title_key:
        continue

    if title_key not in rich_text_title_to_model_row:
        rich_text_title_to_model_row[
            title_key
        ] = int(model_row)

assert rich_text_title_to_model_row, (
    "The rich-text title mapping was not created."
)

print(
    "Unique eligible titles mapped:",
    f"{len(rich_text_title_to_model_row):,}",
)

Unique eligible titles mapped: 85,685


In [16]:
def search_game_titles_rich_text(
    search_text,
    limit=20,
):
    """Search eligible game titles using token-based matching."""

    query_key = normalise_game_title(
        search_text
    )

    query_tokens = set(
        query_key.split()
    )

    if not query_tokens:
        return []

    title_tokens = (
        description_games[
            "normalised_display_name_rich_text"
        ]
        .astype(str)
        .str.split()
    )

    matches = title_tokens.apply(
        lambda tokens: query_tokens.issubset(
            set(tokens)
        )
    )

    return (
        description_games.loc[
            matches,
            "display_name",
        ]
        .drop_duplicates()
        .head(limit)
        .tolist()
    )


search_game_titles_rich_text(
    "assassin creed"
)

["Assassin's Creed® Odyssey",
 "Assassin's Creed® Origins",
 "Assassin's Creed® Unity",
 "Assassin's Creed 2",
 "Assassin's Creed® Syndicate",
 "Assassin's Creed Valhalla",
 'Assassin’s Creed® Brotherhood',
 "Assassin's Creed™: Director's Cut Edition",
 'Assassin’s Creed® Rogue',
 "Assassin's Creed® Revelations",
 'Assassin’s Creed® III',
 "Assassin's Creed® III Remastered",
 "Assassin's Creed Mirage",
 'Assassin’s Creed® Chronicles: China',
 'Assassin’s Creed® Liberation HD',
 "Assassin's Creed Freedom Cry",
 'Assassin’s Creed® Chronicles: Russia',
 'Assassin’s Creed® Chronicles: India']

## 10. Rich-Text Recommendation Function

The function retrieves a candidate pool, removes the query game and duplicate application identifiers, converts cosine distance into similarity, and returns a standardised hybrid-compatible schema.

The textual score is kept separate from popularity and review signals. Those signals are included only as optional fields for later analysis and re-ranking.

In [17]:
def recommend_rich_text_v2(
    game_title,
    top_n=10,
    candidate_pool=50,
):
    """Recommend games using description-only TF-IDF similarity."""

    if top_n < 1:
        raise ValueError(
            "top_n must be at least 1."
        )

    if candidate_pool < top_n:
        raise ValueError(
            "candidate_pool must be greater than or equal to top_n."
        )

    title_key = normalise_game_title(
        game_title
    )

    if title_key not in rich_text_title_to_model_row:
        possible_matches = (
            search_game_titles_rich_text(
                game_title,
                limit=10,
            )
        )

        raise ValueError(
            f"Game title not found or not text-eligible: {game_title}. "
            f"Possible matches: {possible_matches}"
        )

    query_model_row = (
        rich_text_title_to_model_row[
            title_key
        ]
    )

    query_row = description_games.iloc[
        query_model_row
    ]

    query_appid = query_row["appid"]

    neighbour_count = min(
        len(description_games),
        max(
            candidate_pool + 1,
            top_n + 1,
        ),
    )

    distances, indices = (
        rich_text_nn.kneighbors(
            rich_text_matrix[
                query_model_row
            ],
            n_neighbors=neighbour_count,
        )
    )

    recommendations = []
    seen_appids = {query_appid}

    optional_columns = [
        "release_date",
        "release_year",
        "genres",
        "recommendations",
        "pct_pos_total",
        "num_reviews_total",
        "peak_ccu",
    ]

    for candidate_row_index, distance in zip(
        indices.flatten(),
        distances.flatten(),
    ):
        candidate_row = (
            description_games.iloc[
                candidate_row_index
            ]
        )

        candidate_appid = (
            candidate_row["appid"]
        )

        if candidate_appid in seen_appids:
            continue

        seen_appids.add(
            candidate_appid
        )

        recommendation = {
            "appid": candidate_appid,
            "recommended_game": candidate_row[
                "display_name"
            ],
            "rich_text_score": round(
                max(
                    0.0,
                    min(
                        1.0,
                        1 - float(distance),
                    ),
                ),
                6,
            ),
        }

        for column in optional_columns:
            if column in description_games.columns:
                recommendation[column] = (
                    candidate_row[column]
                )

        recommendations.append(
            recommendation
        )

        if len(recommendations) >= candidate_pool:
            break

    result = pd.DataFrame(
        recommendations
    )

    if result.empty:
        return result

    sort_columns = [
        "rich_text_score",
    ]

    ascending = [
        False,
    ]

    if "recommendations" in result.columns:
        sort_columns.append(
            "recommendations"
        )

        ascending.append(
            False
        )

    result = (
        result
        .sort_values(
            sort_columns,
            ascending=ascending,
            na_position="last",
            kind="stable",
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    result.insert(
        0,
        "rich_text_rank",
        np.arange(
            1,
            len(result) + 1,
        ),
    )

    return result

## 11. Recommendation Case Studies

The same four query games used by the metadata recommender are retained for consistent cross-model comparison:

- **ELDEN RING**
- **Red Dead Redemption 2**
- **PUBG: BATTLEGROUNDS**
- **Assassin's Creed Odyssey**

These queries cover Souls-like role-playing, narrative open-world action, battle royale multiplayer, and franchise-heavy action-adventure recommendations.

In [ ]:
test_games = [
    "ELDEN RING",
    "Red Dead Redemption 2",
    "PUBG: BATTLEGROUNDS",
    "Assassin's Creed Odyssey",
]

for game_title in test_games:
    print("\n" + "=" * 80)
    print(
        f"Rich-text recommendations for: {game_title}"
    )
    print("=" * 80)

    display(
        recommend_rich_text_v2(
            game_title=game_title,
            top_n=10,
            candidate_pool=50,
        )
    )


Rich-text recommendations for: ELDEN RING


,rich_text_rank,appid,recommended_game,rich_text_score,release_date,release_year,genres,recommendations,pct_pos_total,num_reviews_total,peak_ccu
0,1,2012170,Ashzel & The Power Dagger,0.180541,2022-06-10,2022,"['Action', 'Adventure', 'Indie', 'RPG', 'Strat...",0,NaN,0.0,0
1,2,3331350,Void Lands,0.117666,2024-11-22,2024,"['Action', 'Indie', 'RPG', 'Early Access']",0,NaN,0.0,0
2,3,3208830,Pixel Dungeon RPG,0.115453,2024-09-27,2024,"['Massively Multiplayer', 'RPG', 'Free To Play']",0,33.0,15.0,0
3,4,3358020,Olympus of the Heavens,0.113465,2025-02-15,2025,"['Action', 'Adventure', 'RPG', 'Early Access']",0,NaN,0.0,0
4,5,2648980,Warmage,0.109426,2024-10-26,2024,"['Indie', 'RPG', 'Strategy']",0,NaN,0.0,0
5,6,3097450,Warmage: Prologue,0.108326,2024-08-05,2024,"['Strategy', 'Free To Play']",0,NaN,0.0,0
6,7,3245250,Of Ink and Silver,0.107844,2025-01-28,2025,"['Adventure', 'Indie', 'RPG', 'Early Access']",0,NaN,0.0,0
7,8,3120270,The Castle Of Xanxillia,0.107274,2024-08-19,2024,"['Action', 'Adventure', 'Indie', 'RPG']",0,NaN,0.0,0
8,9,1546090,RAIDBORN,0.107199,2023-03-29,2023,"['Action', 'Adventure', 'RPG', 'Early Access']",249,75.0,253.0,5
9,10,763890,Wildermyth,0.106566,2021-06-15,2021,"['Indie', 'RPG', 'Strategy']",15549,95.0,15555.0,207



Rich-text recommendations for: Red Dead Redemption 2


,rich_text_rank,appid,recommended_game,rich_text_score,release_date,release_year,genres,recommendations,pct_pos_total,num_reviews_total,peak_ccu
0,1,2668510,Red Dead Redemption,0.284116,2024-10-29,2024,['Action'],8276,92.0,8295.0,763
1,2,1404210,Red Dead Online,0.181455,2020-12-01,2020,"['Action', 'Adventure']",56631,83.0,56651.0,1228
2,3,3260490,Cowgirl Trainer,0.136007,2024-10-19,2024,"['Indie', 'RPG', 'Free To Play']",0,91.0,12.0,0
3,4,3245330,The Final Exam,0.122369,2024-11-01,2024,"['Action', 'Adventure', 'Indie', 'Simulation',...",0,46.0,75.0,0
4,5,1413870,Shadow Man Remastered,0.119234,2021-04-15,2021,"['Action', 'Adventure']",1150,95.0,1151.0,12
5,6,2495340,20 Doors,0.118299,2023-07-21,2023,"['Action', 'Adventure', 'Indie']",0,NaN,0.0,0
6,7,1263120,Hecaton,0.117637,2020-04-05,2020,"['Action', 'Indie', 'RPG']",0,NaN,0.0,0
7,8,1996770,Turok 3: Shadow of Oblivion Remastered,0.117493,2023-11-30,2023,"['Action', 'Adventure']",430,88.0,431.0,4
8,9,340560,Wanted Corp.,0.116386,2016-12-08,2016,['Action'],0,45.0,11.0,0
9,10,1970700,CowboysShowdown,0.115619,2022-05-25,2022,"['Action', 'Casual', 'Indie']",0,90.0,11.0,1



Rich-text recommendations for: PUBG: BATTLEGROUNDS


,rich_text_rank,appid,recommended_game,rich_text_score,release_date,release_year,genres,recommendations,pct_pos_total,num_reviews_total,peak_ccu
0,1,843730,Infected Battlegrounds,0.257611,2018-08-02,2018,"['Action', 'Indie', 'Early Access']",0,50.0,34.0,0
1,2,2630220,HEROS FIGHT Battle royal,0.246799,2023-10-27,2023,"['Action', 'Strategy']",0,NaN,0.0,0
2,3,873220,SurvivalZ Battlegrounds,0.242337,2019-04-02,2019,"['Action', 'Indie', 'Simulation', 'Strategy']",273,64.0,273.0,0
3,4,1148300,War Battle Royale Battlegrounds,0.240717,2019-09-11,2019,"['Action', 'Simulation', 'Strategy']",0,NaN,0.0,0
4,5,1176170,GIRLS BATTLEGROUNDS | 性感大逃杀,0.235731,2019-11-18,2019,"['Action', 'Adventure']",0,57.0,59.0,0
5,6,1172470,Apex Legends™,0.215830,2020-11-04,2020,"['Action', 'Adventure', 'Free To Play']",1548,67.0,983230.0,151844
6,7,1114020,ZomB: Battlegrounds,0.214394,2019-07-26,2019,"['Action', 'Indie']",0,NaN,0.0,0
7,8,658470,Mini Battlegrounds,0.209734,2018-08-20,2018,"['Action', 'Casual', 'Indie', 'Massively Multi...",207,64.0,586.0,0
8,9,805940,RUSSIA BATTLEGROUNDS,0.208386,2018-07-13,2018,"['Action', 'Adventure', 'Indie', 'Massively Mu...",3722,77.0,3723.0,3
9,10,653080,Inflatality,0.200093,2017-10-26,2017,"['Action', 'Casual', 'Indie']",0,66.0,15.0,0



Rich-text recommendations for: Assassin's Creed Odyssey


,rich_text_rank,appid,recommended_game,rich_text_score,release_date,release_year,genres,recommendations,pct_pos_total,num_reviews_total,peak_ccu
0,1,2251680,Epic Assassin,0.298995,2022-12-30,2022,['RPG'],0,NaN,0.0,0
1,2,202690,Hegemony Gold: Wars of Ancient Greece,0.197679,2012-03-30,2012,"['Indie', 'Strategy']",176,94.0,177.0,5
2,3,1351200,Museum of War,0.192201,2020-07-28,2020,"['Action', 'Indie']",0,NaN,0.0,0
3,4,383740,Marble Age: Remastered,0.189095,2020-11-03,2020,"['Indie', 'Simulation', 'Strategy']",330,85.0,330.0,5
4,5,1452870,Roads of Time 2: Odyssey,0.169379,2021-04-08,2021,['Casual'],0,72.0,11.0,0
5,6,1355070,Aenaon,0.160532,2020-07-26,2020,"['Adventure', 'Early Access']",0,NaN,0.0,0
6,7,540690,Hellenica,0.152317,2017-01-23,2017,"['Indie', 'RPG', 'Strategy']",0,65.0,44.0,0
7,8,1191280,12 Labours of Hercules X: Greed for Speed,0.145766,2020-04-22,2020,"['Adventure', 'Casual', 'Indie', 'Strategy']",0,71.0,21.0,3
8,9,726630,Mare Nostrvm,0.142233,2017-11-02,2017,['Strategy'],0,85.0,21.0,1
9,10,855390,Ryte - The Eye of Atlantis,0.138745,2021-01-27,2021,['Adventure'],0,50.0,24.0,0


## 12. Preparing Rich-Text Candidates for the Hybrid Recommender

For each case-study query, the Top 50 rich-text candidates are generated using a standardised schema.

The saved CSV is an experimental handoff for hybrid development and evaluation. In the final live application, `recommend_rich_text_v2()` can be called directly for any eligible catalogue game.

In [ ]:
def generate_rich_text_candidates(
    query_title,
    top_n=50,
    candidate_pool=100,
):
    """Generate a hybrid-compatible rich-text candidate table."""

    results = recommend_rich_text_v2(
        game_title=query_title,
        top_n=top_n,
        candidate_pool=candidate_pool,
    ).copy()

    title_key = normalise_game_title(
        query_title
    )

    query_model_row = (
        rich_text_title_to_model_row[
            title_key
        ]
    )

    query_row = description_games.iloc[
        query_model_row
    ]

    results.insert(
        0,
        "query_appid",
        query_row["appid"],
    )

    results.insert(
        1,
        "query_game",
        query_row["display_name"],
    )

    return results

In [ ]:
rich_text_candidate_tables = []

for game_title in test_games:
    print(
        f"Generating rich-text candidates for: {game_title}"
    )

    candidates = generate_rich_text_candidates(
        query_title=game_title,
        top_n=50,
        candidate_pool=100,
    )

    rich_text_candidate_tables.append(
        candidates
    )

rich_text_candidates = pd.concat(
    rich_text_candidate_tables,
    ignore_index=True,
)

print(
    "Combined rich-text candidate shape:",
    rich_text_candidates.shape,
)

display(
    rich_text_candidates[
        [
            "query_game",
            "rich_text_rank",
            "appid",
            "recommended_game",
            "rich_text_score",
        ]
    ]
    .head(20)
)

Generating rich-text candidates for: ELDEN RING
Generating rich-text candidates for: Red Dead Redemption 2
Generating rich-text candidates for: PUBG: BATTLEGROUNDS
Generating rich-text candidates for: Assassin's Creed Odyssey
Combined rich-text candidate shape: (200, 13)


,query_game,rich_text_rank,appid,recommended_game,rich_text_score
0,ELDEN RING,1,2012170,Ashzel & The Power Dagger,0.180541
1,ELDEN RING,2,3331350,Void Lands,0.117666
2,ELDEN RING,3,3208830,Pixel Dungeon RPG,0.115453
3,ELDEN RING,4,3358020,Olympus of the Heavens,0.113465
4,ELDEN RING,5,2648980,Warmage,0.109426
5,ELDEN RING,6,3097450,Warmage: Prologue,0.108326
6,ELDEN RING,7,3245250,Of Ink and Silver,0.107844
7,ELDEN RING,8,3120270,The Castle Of Xanxillia,0.107274
8,ELDEN RING,9,1546090,RAIDBORN,0.107199
9,ELDEN RING,10,763890,Wildermyth,0.106566


## 13. Candidate Validation and Export

The hybrid handoff is validated before export. Every query must contain 50 candidates, with no missing identifiers, duplicate query-candidate pairs, self-recommendations, invalid scores or invalid ranks.

In [ ]:
query_counts = (
    rich_text_candidates
    .groupby("query_game")
    .size()
)

duplicate_pairs = (
    rich_text_candidates
    .duplicated(
        subset=[
            "query_appid",
            "appid",
        ]
    )
    .sum()
)

missing_appids = (
    rich_text_candidates["appid"]
    .isna()
    .sum()
)

self_recommendations = (
    rich_text_candidates["query_appid"]
    .eq(
        rich_text_candidates["appid"]
    )
    .sum()
)

invalid_scores = (
    ~rich_text_candidates[
        "rich_text_score"
    ]
    .between(
        0,
        1,
        inclusive="both",
    )
).sum()

invalid_ranks = 0

for _, group in rich_text_candidates.groupby(
    "query_game"
):
    expected_ranks = list(
        range(
            1,
            len(group) + 1,
        )
    )

    actual_ranks = (
        group
        .sort_values(
            "rich_text_rank"
        )[
            "rich_text_rank"
        ]
        .tolist()
    )

    if actual_ranks != expected_ranks:
        invalid_ranks += 1

expected_rows = len(test_games) * 50

print("Candidates per query:")
print(query_counts)

print(
    "\nCombined candidate shape:",
    rich_text_candidates.shape,
)

print(
    "Duplicate query-candidate pairs:",
    duplicate_pairs,
)

print(
    "Missing candidate appids:",
    missing_appids,
)

print(
    "Self-recommendations:",
    self_recommendations,
)

print(
    "Invalid scores:",
    invalid_scores,
)

print(
    "Query groups with invalid ranks:",
    invalid_ranks,
)

assert len(rich_text_candidates) == expected_rows
assert (query_counts == 50).all()
assert duplicate_pairs == 0
assert missing_appids == 0
assert self_recommendations == 0
assert invalid_scores == 0
assert invalid_ranks == 0

print(
    "\nRich-text candidate validation passed."
)

Candidates per query:
query_game
Assassin's Creed® Odyssey    50
ELDEN RING                   50
PUBG: BATTLEGROUNDS          50
Red Dead Redemption 2        50
dtype: int64

Combined candidate shape: (200, 13)
Duplicate query-candidate pairs: 0
Missing candidate appids: 0
Self-recommendations: 0
Invalid scores: 0
Query groups with invalid ranks: 0

Rich-text candidate validation passed.


In [ ]:
output_directory = (
    "/content/drive/MyDrive/"
    "MSC_DISSERTATION/Output"
)

os.makedirs(
    output_directory,
    exist_ok=True,
)

candidate_output_path = os.path.join(
    output_directory,
    "rich_text_candidates.csv",
)

config_output_path = os.path.join(
    output_directory,
    "rich_text_model_config.json",
)

rich_text_candidates.to_csv(
    candidate_output_path,
    index=False,
)

rich_text_model_config = {
    "model_name": (
        "Description-based rich-text TF-IDF recommender"
    ),
    "primary_text_field": (
        "about_the_game_clean"
    ),
    "fallback_text_field": (
        "short_description_clean"
    ),
    "minimum_description_words": (
        MIN_DESCRIPTION_WORDS
    ),
    "max_features": (
        RICH_TEXT_TFIDF_CONFIG[
            "max_features"
        ]
    ),
    "ngram_range": list(
        RICH_TEXT_TFIDF_CONFIG[
            "ngram_range"
        ]
    ),
    "min_df": (
        RICH_TEXT_TFIDF_CONFIG[
            "min_df"
        ]
    ),
    "max_df": (
        RICH_TEXT_TFIDF_CONFIG[
            "max_df"
        ]
    ),
    "sublinear_tf": (
        RICH_TEXT_TFIDF_CONFIG[
            "sublinear_tf"
        ]
    ),
    "distance_metric": "cosine",
    "nearest_neighbour_algorithm": "brute",
    "candidate_pool": 100,
    "exported_candidates_per_query": 50,
    "case_study_queries": test_games,
}

with open(
    config_output_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        rich_text_model_config,
        file,
        indent=4,
    )

print(
    "Rich-text candidates saved to:\n"
    f"{candidate_output_path}"
)

print(
    "\nModel configuration saved to:\n"
    f"{config_output_path}"
)

Rich-text candidates saved to:
/content/drive/MyDrive/MSC_DISSERTATION/Output/rich_text_candidates.csv

Model configuration saved to:
/content/drive/MyDrive/MSC_DISSERTATION/Output/rich_text_model_config.json


In [ ]:
saved_rich_text_candidates = pd.read_csv(
    candidate_output_path
)

with open(
    config_output_path,
    "r",
    encoding="utf-8",
) as file:
    saved_rich_text_config = json.load(
        file
    )

print(
    "Saved candidate shape:",
    saved_rich_text_candidates.shape,
)

display(
    saved_rich_text_candidates.head(10)
)

saved_rich_text_config

Saved candidate shape: (200, 13)


,query_appid,query_game,rich_text_rank,appid,recommended_game,rich_text_score,release_date,release_year,genres,recommendations,pct_pos_total,num_reviews_total,peak_ccu
0,1245620,ELDEN RING,1,2012170,Ashzel & The Power Dagger,0.180541,2022-06-10,2022,"['Action', 'Adventure', 'Indie', 'RPG', 'Strat...",0,NaN,0.0,0
1,1245620,ELDEN RING,2,3331350,Void Lands,0.117666,2024-11-22,2024,"['Action', 'Indie', 'RPG', 'Early Access']",0,NaN,0.0,0
2,1245620,ELDEN RING,3,3208830,Pixel Dungeon RPG,0.115453,2024-09-27,2024,"['Massively Multiplayer', 'RPG', 'Free To Play']",0,33.0,15.0,0
3,1245620,ELDEN RING,4,3358020,Olympus of the Heavens,0.113465,2025-02-15,2025,"['Action', 'Adventure', 'RPG', 'Early Access']",0,NaN,0.0,0
4,1245620,ELDEN RING,5,2648980,Warmage,0.109426,2024-10-26,2024,"['Indie', 'RPG', 'Strategy']",0,NaN,0.0,0
5,1245620,ELDEN RING,6,3097450,Warmage: Prologue,0.108326,2024-08-05,2024,"['Strategy', 'Free To Play']",0,NaN,0.0,0
6,1245620,ELDEN RING,7,3245250,Of Ink and Silver,0.107844,2025-01-28,2025,"['Adventure', 'Indie', 'RPG', 'Early Access']",0,NaN,0.0,0
7,1245620,ELDEN RING,8,3120270,The Castle Of Xanxillia,0.107274,2024-08-19,2024,"['Action', 'Adventure', 'Indie', 'RPG']",0,NaN,0.0,0
8,1245620,ELDEN RING,9,1546090,RAIDBORN,0.107199,2023-03-29,2023,"['Action', 'Adventure', 'RPG', 'Early Access']",249,75.0,253.0,5
9,1245620,ELDEN RING,10,763890,Wildermyth,0.106566,2021-06-15,2021,"['Indie', 'RPG', 'Strategy']",15549,95.0,15555.0,207


{'model_name': 'Description-based rich-text TF-IDF recommender',
 'primary_text_field': 'about_the_game_clean',
 'fallback_text_field': 'short_description_clean',
 'minimum_description_words': 30,
 'max_features': 30000,
 'ngram_range': [1, 2],
 'min_df': 3,
 'max_df': 0.95,
 'sublinear_tf': True,
 'distance_metric': 'cosine',
 'nearest_neighbour_algorithm': 'brute',
 'candidate_pool': 100,
 'exported_candidates_per_query': 50,
 'case_study_queries': ['ELDEN RING',
  'Red Dead Redemption 2',
  'PUBG: BATTLEGROUNDS',
  "Assassin's Creed Odyssey"]}

## 14. Findings and Limitations

The description-based TF-IDF model provides an interpretable lexical similarity signal. It is particularly useful when games share distinctive terminology, repeated franchise language, mechanics or setting descriptions.

However, TF-IDF primarily measures word and phrase overlap. Semantically similar games may receive weak similarity scores when their descriptions use different vocabulary. Conversely, games that share promotional language may appear closer than their underlying gameplay warrants.

The model also depends on description quality and completeness. Although the minimum-word threshold removes extremely sparse records, catalogue descriptions remain heterogeneous in length, detail and marketing style.

These limitations are not treated as model failure. The rich-text recommender is retained as a complementary candidate generator whose lexical signal can be balanced against structured metadata and graph-based relationships in the final hybrid system.

## 15. Conclusion

This notebook implemented a description-based rich-text recommender using the cleaned Steam Games Dataset 2025.

A non-duplicated descriptive representation was created by using `about_the_game_clean` as the primary field and `short_description_clean` as a fallback. Games with fewer than 30 descriptive words were excluded from model fitting. The eligible descriptions were represented using TF-IDF unigram and bigram features, and cosine nearest neighbours were used to retrieve lexically similar games.

The final recommendation function returns a standardised ranked output with Steam application identifiers and rich-text similarity scores. Top 50 candidate lists were generated for the four shared case-study games and exported as `rich_text_candidates.csv`.

This CSV will be combined with metadata and graph candidate lists in the hybrid recommender. The four case-study queries support development and evaluation only; the reusable recommendation function can process any text-eligible catalogue game.